Copyright (c) 2023 Habana Labs, Ltd. an Intel Company.

#### Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License. You may obtain a copy of the License at https://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

# Using Paramater Efficient Fine Tuning on Llama 2 with 7B Parameters on One Intel&reg; Gaudi&reg; 2 AI Accelerator
This example will Fine Tune the Llama2-7B model using Parameter Efficient Fine Tuining (PEFT) and then run inference on a text prompt.  This will be using the Llama2 model with two task examples from the Optimum Habana library on the Hugging Face model repository.   The Optimum Habana library is optimized for Deep Learning training and inference on First-gen Gaudi and Gaudi2 and offers tasks such as text generation, language modeling, question answering and more. For all the examples and models, please refer to the [Optimum Habana GitHub](https://github.com/huggingface/optimum-habana#validated-models).

This example will Fine Tune the Llama2-7B model using Parameter Efficient Fine Tuining (PEFT) on the timdettmers/openassistant-guanaco dataset using the Language-Modeling Task in Optimum Habana.

### Parameter Efficient Fine Tuning with Low Rank Adaptation
Parameter Efficient Fine Tuning is a strategy for adapting large pre-trained language models to specific tasks while minimizing computational and memory demands.   It aims to reduce the computational cost and memory requirements associated with fine-tuning large models while maintaining or even improving their performance.  It does so by adding a smaller task-specific layer, leveraging knowledge distillation, and often relying on few-shot learning, resulting in efficient yet effective models for various natural language understanding tasks.   PEFT starts with a pre-trained language model that has already learned a wide range of language understanding tasks from a large corpus of text data. These models are usually large and computationally expensive.   Instead of fine-tuning the entire pre-trained model, PEFT adds a task-specific layer or a few task-specific layers on top of the pre-trained model. These additional layers are relatively smaller and have fewer parameters compared to the base model.


In [17]:
# TODO: upgrade datasets and huggingface_hub
!pip install --upgrade datasets huggingface_hub

Defaulting to user installation because normal site-packages is not writeable
datasets: 2.19.2
huggingface_hub: 0.22.2


In [18]:
# TODO: restart the kernel for the changes to take effect
exit()

In [3]:
# TODO: check that datasets and huggingface_hub are updated to 3.2.0 and 0.28.1, respectively
import datasets, huggingface_hub
print("datasets: %s" %datasets.__version__)
print("huggingface_hub: %s" %huggingface_hub.__version__)

datasets: 3.2.0
huggingface_hub: 0.28.1


In [4]:
%cd ~/Gaudi-tutorials/PyTorch/Single_card_tutorials

/home/u40d817240514e7ee4d7ddc9c6adad24/Gaudi-tutorials/PyTorch/Single_card_tutorials


### Model Setup: 

##### Install the Parameter Efficient Fine Tuning Library methods
This is taking the PEFT method from the Hugging Face repository and will be used to help create the PEFT Fine Tuning with the Llama2 model.

In [5]:
import sys
!{sys.executable} -m pip install peft==0.10.0

Defaulting to user installation because normal site-packages is not writeable


##### Install the Optimum-Habana Library

In [6]:
!{sys.executable} -m pip install -q optimum-habana==1.11.1

##### Pull the Hugging Face Examples from GitHub
These contain the working Hugging Face Task Examples that have been optimized for Gaudi.  For Fine Tuning, we'll use the language-modeling task. 

In [7]:
%cd ~/Gaudi-tutorials/PyTorch/Single_card_tutorials

/home/u40d817240514e7ee4d7ddc9c6adad24/Gaudi-tutorials/PyTorch/Single_card_tutorials


In [8]:
# TODO: change to version 1.11.1
#!git clone -b v1.12.0 https://github.com/huggingface/optimum-habana.git
!git clone -b v1.11.1 https://github.com/huggingface/optimum-habana.git

fatal: destination path 'optimum-habana' already exists and is not an empty directory.


##### Go to the Language Modeling Task and install the model specific requirements

In [9]:
%cd ~/Gaudi-tutorials/PyTorch/Single_card_tutorials/optimum-habana/examples/language-modeling

/home/u40d817240514e7ee4d7ddc9c6adad24/Gaudi-tutorials/PyTorch/Single_card_tutorials/optimum-habana/examples/language-modeling


In [10]:
# TODO: change peft==0.10.0 in requirements.txt before running
!pip install -q -r requirements.txt

In [11]:
# TODO: Check that peft version is 0.10.0
import peft
peft.__version__

/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:462: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:319: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


'0.10.0'

##### How to access and Use the Llama 2 model

Use of the pretrained model is subject to compliance with third party licenses, including the “Llama 2 Community License Agreement” (LLAMAV2). For guidance on the intended use of the LLAMA2 model, what will be considered misuse and out-of-scope uses, who are the intended users and additional terms please review and read the instructions in this link https://ai.meta.com/llama/license/.
Users bear sole liability and responsibility to follow and comply with any third party licenses, and Habana Labs disclaims and will bear no liability with respect to users’ use or compliance with third party licenses.

To be able to run gated models like the Meta Llama model, you need the following: 
- Have a HuggingFace account
- Agree to the terms of use of the model in its model card on the HF Hub
- set a read token
- Login to your account using the HF CLI: run huggingface-cli login before launching your script

In [ ]:
# TODO: use provided token
!huggingface-cli login --token <insert-token-here>

## Fine Tuning the model with PEFT and LoRA

We'll now run the fine tuning with the PEFT method. Remember that the PEFT methods only fine-tune a small number of extra model parameters, thereby greatly decreasing the computational and storage costs. Recent State-of-the-Art PEFT techniques achieve performance comparable to that of full fine-tuning.

##### Here's a summary of the command required to run the Fine Tuning, you'll run this in the next cell below. 
Note in this case the following: 
1. Using the language modeling with LoRA; `run_lora_clm.py`
2. It's very efficient: only 0.06% of the total paramters are being fine tuned of the total 7B parameters.
4. Only 3 epochs are needed for fine tuning, it takes less than 20 minutes to run with the openassisant-guanaco dataset.
5. We are using a cached model on the Intel Tiber Cloud to reduce download time.


In [13]:
import os
os.environ['HABANA_LOGS'] = '~/logs/habana_logs/'

In [14]:
!python3 run_lora_clm.py \
    --overwrite_output_dir=True \
    --model_name_or_path meta-llama/Llama-2-7b-hf \
    --dataset_name timdettmers/openassistant-guanaco \
    --bf16 True \
    --output_dir ~/Gaudi-tutorials/PyTorch/Single_card_tutorials/model_lora_llama_single \
    --num_train_epochs 3 \
    --per_device_train_batch_size 16 \
    --evaluation_strategy "no" \
    --save_strategy "no" \
    --learning_rate 1e-4 \
    --warmup_ratio  0.03 \
    --lr_scheduler_type "constant" \
    --max_grad_norm  0.3 \
    --logging_steps 1 \
    --do_train \
    --do_eval \
    --use_habana \
    --use_lazy_mode \
    --throughput_warmup_steps 3 \
    --lora_rank=8 \
    --lora_alpha=16 \
    --lora_dropout=0.05 \
    --lora_target_modules "q_proj" "v_proj" \
    --dataset_concatenation \
    --report_to none \
    --max_seq_length 512 \
    --low_cpu_mem_usage True \
    --validation_split_percentage 4 \
    --adam_epsilon 1e-08

/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:462: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:319: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:319: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
02/13/2025 22:54:17 - WARNING - __main__ -   Process rank: 0, device: hpu, distributed training: True, 16-bits training: True
02/13/2025 22:54:17 - INFO - __mai

#### LoRA Fine Tuning Completed
You will now see a "model_lora_llama_single" folder created which contains the PEFT model `adapter_model.bin` which will be used in the inference example below. 

## Inference with Llama 2

We'll now use the Hugging Face `text-generation` task to run inference on the Llama2-70b model; we'll generate text based on an included prompt.  Notice that we've included a path to the PEFT model that we just created.

First, we'll move to the text-generation examples folder and install the requirements. 

In [15]:
%cd ~/Gaudi-tutorials/PyTorch/Single_card_tutorials/optimum-habana/examples/text-generation
!pip install -q -r requirements.txt

/home/u40d817240514e7ee4d7ddc9c6adad24/Gaudi-tutorials/PyTorch/Single_card_tutorials/optimum-habana/examples/text-generation


You will see that we are now running inference with the `run_generation.py` task and we are including the PEFT model that we Fine Tuned in the steps above. 

```
python3 run_generation.py \
--model_name_or_path meta-llama/Llama-2-7b-hf \
--batch_size 1 \
--do_sample
--max_new_tokens 250 \
--n_iterations 4
--use_hpu_graphs \
--use_kv_cache \
--bf16 \
--prompt "I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don't forget to order a big bone-shaped cake for me to share with my fur friends!" \
--peft_model /root/Gaudi-tutorials/PyTorch/Single_card_tutorials/optimum-habana/examples/language-modeling/model_lora_llama_single
```

In [16]:
prompt = input("Enter a prompt for text generation: ")

Enter a prompt for text generation:  I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don't forget to order a big bone-shaped cake for me to share with my fur friends!


In [17]:
cmd = f'python3 run_generation.py  --model_name_or_path meta-llama/Llama-2-7b-hf --batch_size 1 --do_sample --max_new_tokens 300 --n_iterations 4 \
      --use_hpu_graphs --use_kv_cache --bf16 --prompt "{prompt}" \
      --peft_model ~/Gaudi-tutorials/PyTorch/Single_card_tutorials/model_lora_llama_single '
print(cmd)
import os
os.system(cmd)

python3 run_generation.py  --model_name_or_path meta-llama/Llama-2-7b-hf --batch_size 1 --do_sample --max_new_tokens 300 --n_iterations 4       --use_hpu_graphs --use_kv_cache --bf16 --prompt "I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don't forget to order a big bone-shaped cake for me to share with my fur friends!"       --peft_model ~/Gaudi-tutorials/PyTorch/Single_card_tutorials/model_lora_llama_single 


/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:462: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:319: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:319: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 3283.21it/s]
02/13/2025 23:20:30 - INFO - __main__ - Single-device run.
/home/u40d817240514e7ee4d7ddc9c6ada

Warming up
Warming up
Warming up


02/13/2025 23:20:55 - INFO - __main__ - Running generate...



Input/outputs:
input 1: ("I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don't forget to order a big bone-shaped cake for me to share with my fur friends!",)
output 1: ("I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don't forget to order a big bone-shaped cake for me to share with my fur friends!### Assistant: Sure! Here are some ideas for planning a surprise birthday party for your human:\n\nDecorations: You can use balloons, streamers, and banners to create a festive atmosphere. You can also use your human's favorite colors or pictures to decorate the room.\n\nGames: You can play a game of fetch or hide-and-seek with your human and their friends. You can also organize a treasure hunt or scavenger hunt for your human to solve.\n\nFood and drinks: You can arrange for your human's favorite food and drinks to be served at the party.

0

###### Inference Output with PEFT

```
input 1: ("I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don't forget to order a big bone-shaped cake for me to share with my fur friends!",)
output 1: ('I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don\'t forget to order a big bone-shaped cake for me to share with my fur friends!

Assistant: Hey there pup! I can help you plan your human\'s birthday party. Here are some ideas for fun activities and games you can play together:\n\n
1. A "Find the Treat" scavenger hunt: Hide treats around your home or yard for your human to find. Provide clues and hints along the way.\n
2. "Tug-of-War": Play a game of tug-of-war with a rope tied to a tree stump or post.\n
3. "Frisbee Fun": Invite your human to a game of fetch with a Frisbee in the park or backyard.\n\n
Decorations can include: Dog-shaped balloons, paw print streamers, and a banner saying "Happy Birthday" with your human\'s name.\n\n
And don\'t forget to order a cake in the shape of a big bone for you and your fur friends to share!
```

##### Comparison without PEFT and LoRA
In this example, we're simply running the Llama2 7B model **without** including the PEFT fine tuned model, so the you are losing the additional detail that is brought to the model, and the results have signficantly less information and fidelity compared to the last model.

In [18]:
cmd = f'python3 run_generation.py  --model_name_or_path meta-llama/Llama-2-7b-hf --batch_size 1 --do_sample --max_new_tokens 300 --n_iterations 4 \
      --use_hpu_graphs --use_kv_cache --bf16 --prompt "{prompt}"'
print(cmd)
import os
os.system(cmd)

python3 run_generation.py  --model_name_or_path meta-llama/Llama-2-7b-hf --batch_size 1 --do_sample --max_new_tokens 300 --n_iterations 4       --use_hpu_graphs --use_kv_cache --bf16 --prompt "I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don't forget to order a big bone-shaped cake for me to share with my fur friends!"


/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:462: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:319: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/u40d817240514e7ee4d7ddc9c6adad24/.local/lib/python3.10/site-packages/transformers/utils/generic.py:319: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 2817.81it/s]
02/13/2025 23:35:35 - INFO - __main__ - Single-device run.
/home/u40d817240514e7ee4d7ddc9c6ada

Warming up
Warming up
Warming up


02/13/2025 23:35:58 - INFO - __main__ - Running generate...



Input/outputs:
input 1: ("I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don't forget to order a big bone-shaped cake for me to share with my fur friends!",)
output 1: ('I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don\'t forget to order a big bone-shaped cake for me to share with my fur friends!\n"It\'s not just a game, it\'s a lifestyle."\n"The only thing better than a game is a game with your friends."\n"It\'s not a game, it\'s a lifestyle."\n"It\'s not just a game, it\'s a way of life."\n"It\'s not just a game, it\'s a way of life."\n"It\'s not just a game, it\'s a way of life."\n"It\'s not just a game, it\'s a lifestyle."\n"It\'s not just a game, it\'s a lifestyle."\n"It\'s not just a game, it\'s a way of life."\n"It\'s not just a game, it\'s a way of life."\n"It\'s not just a game, it\'s a lifestyle."\n"It\'s not just a gam

0

###### Inference Output without PEFT (using just standard Llama 2 model)

```
input 1: ("I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don't forget to order a big bone-shaped cake for me to share with my fur friends!",)
output 1: ("I am a dog. Please help me plan a surprise birthday party for my human, including fun activities, games and decorations. And don't forget to order a big bone-shaped cake for me to share with my fur friends!\n

Make sure that you do not make a big noise because my human doesn’t know that we are planning a birthday party. Thanks to your help now I am sure there are no more things to worry about.\n
The dog does not have to worry that the human will find out about the party. She should not worry about the noise while planning the party. There will be big bone-shaped cake for the guest of honor to share with his fur friends. There will be fun activities, games and decorations. The following items are tagged newsletter marketing:\n
```

In [ ]:
exit()